In [0]:

# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Bronze
# Notebook        : bronze_purchase_orders
# Source          : purchase_orders.csv
# Target          : procurement.bronze.bronze_purchase_orders
# Audit Table     : procurement.audit.duplicate_purchase_orders
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads raw purchase_orders master data into the Bronze layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Load purchase_orders master data from the source CSV file into the Bronze layer.
#
# Preserve the raw source data with minimal transformations.
#
# Detect duplicate Payment IDs and store them in the Audit schema for business review.
#
# Add audit columns to support data lineage and traceability.
#
# Create a reliable Bronze Delta table that will serve as the source for the Silver layer.
#
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
print(CATALOG)
print(BRONZE_PURCHASE_ORDERS)
print(AUDIT_DUPLICATE_PURCHASE_ORDERS)
print(PURCHASE_ORDERS_FILE)

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
#Purchase_Orders schema
purchase_orders_schema = StructType([
    StructField("po_id", StringType(), False),
    StructField("po_date", StringType(), True),
    StructField("pr_id", StringType(), True),
    StructField("supplier_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity_ordered", IntegerType(), True),
    StructField("unit_price", DecimalType(18,2), True),
    StructField("total_amount", DecimalType(18,2), True),
    StructField("currency",StringType(),True),
    StructField("po_status", StringType(), True) ,
    StructField("is_duplicate_po", StringType(), True) ,
    StructField("expected_delivery_date", StringType(), True) ,
    StructField("payment_terms", StringType(), True)
  
])
# Read Purchase_Orders master data from landing volume
bronze_purchase_orders_df = (spark.read
    .format("csv")
    .option("header", True)
    .schema(purchase_orders_schema)
    .load(PURCHASE_ORDERS_FILE)
)

#Source Data validation 
print(f"Total Records : {bronze_purchase_orders_df.count()}")

print("\nSchema:")
bronze_purchase_orders_df.printSchema()

print("\nColumns:")
print(bronze_purchase_orders_df.columns)

print("\nSampledata:")
display(bronze_purchase_orders_df.limit(10))

In [0]:
# Check the NULL and Blank payment_ids
null_blank_po_id = bronze_purchase_orders_df.filter(col("po_id").isNull() | (trim(col("po_id")) == ""))

print(f"Total NULL or Blank po_ids : {null_blank_po_id.count()}")

display(null_blank_po_id)

In [0]:
#Check the NULL and Blank po date
null_blank_po_date = bronze_purchase_orders_df.filter(col("po_date").isNull() | (trim(col("po_date")) == ""))

print(f"Total NULL or Blank po_date : {null_blank_po_date.count()}")

display(null_blank_po_date)

In [0]:
# Check the NULL and Blank pr_ids
null_blank_pr_id = bronze_purchase_orders_df.filter(col("pr_id").isNull() | (trim(col("pr_id")) == ""))

print(f"Total NULL or Blank pr_ids : {null_blank_pr_id.count()}")

display(null_blank_pr_id)

In [0]:
# Check the NULL and Blank supplier_ids
null_blank_supplier_id = bronze_purchase_orders_df.filter(col("supplier_id").isNull() | (trim(col("supplier_id")) == ""))

print(f"Total NULL or Blank supplier_ids : {null_blank_supplier_id.count()}")

display(null_blank_supplier_id)

In [0]:
# Check the NULL and Blank product_ids
null_blank_product_id = bronze_purchase_orders_df.filter(col("product_id").isNull() | (trim(col("product_id")) == ""))

print(f"Total NULL or Blank product_ids : {null_blank_product_id.count()}")

display(null_blank_product_id)

In [0]:
# Check the NULL and Blank quantity_ordered
null_blank_quantity_ordered = bronze_purchase_orders_df.filter(col("quantity_ordered").isNull() | (trim(col("quantity_ordered")) == ""))

print(f"Total NULL or Blank product_ids : {null_blank_quantity_ordered.count()}")

display(null_blank_quantity_ordered)

In [0]:
#Check the NUll and negative unitprice
negative_unit_price = bronze_purchase_orders_df.filter((col("unit_price") < 0) | (col("unit_price").isNull()))

print(f"Total Null and negative amounts : {negative_unit_price.count()}")

display(negative_unit_price)

In [0]:
#Check the NUll and negative total_amount
negative_total_amount = bronze_purchase_orders_df.filter((col("total_amount") < 0) | (col("total_amount").isNull()))

print(f"Total Null and negative amounts : {negative_total_amount.count()}")

display(negative_total_amount)

In [0]:
#Check the NULL and Blank currency
null_blank_currency = bronze_purchase_orders_df.filter(col("currency").isNull() | (trim(col("currency")) == ""))

print(f"Total NULL or Blank currency : {null_blank_currency.count()}")

display(null_blank_currency)

In [0]:
#Check the NULL and Blank po_status
null_blank_po_status = bronze_purchase_orders_df.filter(col("po_status").isNull() | (trim(col("po_status")) == ""))

print(f"Total NULL or Blank po_status : {null_blank_po_status.count()}")

display(null_blank_po_status)

In [0]:
#Check the NULL and Blank is_duplicate_po
null_blank_is_duplicate_po = bronze_purchase_orders_df.filter(col("is_duplicate_po").isNull() | (trim(col("is_duplicate_po")) == ""))

print(f"Total NULL or Blank is_duplicate_po : {null_blank_is_duplicate_po.count()}")

display(null_blank_is_duplicate_po)

In [0]:
#Check the NULL and Blank expected_delivery_date
null_blank_expected_delivery_date = bronze_purchase_orders_df.filter(col("expected_delivery_date").isNull() | (trim(col("expected_delivery_date")) == ""))

print(f"Total NULL or Blank expected_delivery_date : {null_blank_expected_delivery_date.count()}")

display(null_blank_is_duplicate_po)

In [0]:
#Check the NULL and Blank payment_terms
null_blank_payment_terms = bronze_purchase_orders_df.filter(col("payment_terms").isNull() | (trim(col("payment_terms")) == ""))

print(f"Total NULL or Blank payment_terms : {null_blank_payment_terms.count()}")

display(null_blank_payment_terms)

In [0]:
# ============================================================
# Identify Duplicate purchase_orders IDs
# ============================================================

duplicate_purchase_orders_keys = (
    bronze_purchase_orders_df
        .groupBy("po_id")
        .count()
        .filter(col("count") > 1)
        .withColumnRenamed("count", "duplicate_IDs")
)

display(duplicate_purchase_orders_keys)

In [0]:
# ============================================================
# Identify Duplicate purchase_orders Records
# Business Rule: Keep the first occurrence of each PO ID and identify subsequent records as duplicates.
# ============================================================

window_spec = Window.partitionBy("po_id").orderBy("po_id")

purchase_orders_rank_df = (
    bronze_purchase_orders_df
        .join(
            duplicate_purchase_orders_keys.select("po_id"),
            on="po_id",
            how="inner"
        )
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
)
display(purchase_orders_rank_df)

In [0]:
# ============================================================
# Retrieve Duplicate  Payment Records
# ============================================================

duplicate_purchase_orders = (
    purchase_orders_rank_df
        .filter(col("row_num") > 1)
        .drop("row_num")
)

print(f"Duplicate Purchase_Orders Records : {duplicate_purchase_orders.count()}")
display(duplicate_purchase_orders)


In [0]:
# ============================================================
# Add Audit Metadata for duplicate PO IDs
# ============================================================

duplicate_purchase_orders = (
    duplicate_purchase_orders
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("Purchase_Orders"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(duplicate_purchase_orders)

In [0]:
# ============================================================
# Add Audit Metadata for NULL PO IDs
# ============================================================

from pyspark.sql.functions import current_timestamp, lit

invalid_purchase_orders = (
    null_blank_po_id
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("employees"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(invalid_purchase_orders)

In [0]:
# ============================================================
# Write Duplicate Records to Audit Table
# ============================================================

duplicate_count = duplicate_purchase_orders.count()

if duplicate_count > 0:

    write_delta(
        df = duplicate_purchase_orders,
         table_name = AUDIT_DUPLICATE_PURCHASE_ORDERS
    )

    print(f"Successfully written {duplicate_count} duplicate record(s) to {AUDIT_DUPLICATE_PURCHASE_ORDERS}")

else:

    print("No duplicate Purchase_Orders records found. Audit table not created.")

In [0]:
# ============================================================
# Write Invalid NULL Records to Audit Table
# ============================================================

invalid_count = invalid_purchase_orders.count()

if invalid_count > 0:

    write_delta(
        df = invalid_purchase_orders,
         table_name = AUDIT_INVALID_PURCHASE_ORDERS
    )

    print(f"Successfully written {invalid_count} invalid record(s) to {AUDIT_INVALID_PURCHASE_ORDERS}")

else:

    print("No invalid purchase_orders records found. Audit table not created.")

In [0]:
# ============================================================
# Add Bronze Audit Columns
# ============================================================

bronze_purchase_orders_final_df = (
    bronze_purchase_orders_df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", lit("Purchase_Orders.csv"))
)
display(bronze_purchase_orders_final_df)

In [0]:
# ============================================================
# Write Bronze Delta Table
# ============================================================

write_delta(df=bronze_purchase_orders_final_df,table_name=BRONZE_PURCHASE_ORDERS)

In [0]:
# ============================================================
# Validate Bronze Delta Table
# ============================================================

bronze_purchase_orders = spark.table(BRONZE_PURCHASE_ORDERS)

print(f"Total Bronze Records : {bronze_purchase_orders.count()}")

display(bronze_purchase_orders)

In [0]:
# ============================================================
# Bronze purchase_orders complete summary
# ============================================================
print("=" * 60)
print("Bronze Purchase_Orders  Load Completed Successfully")
print("=" * 60)

print(f"{'Landing Records':<30}: {bronze_purchase_orders_df.count()}")

print(f"{'Duplicate Audit Records':<30}: {duplicate_purchase_orders.count()}")

print(f"{'Invalid Invoice Records':<30}: {invalid_purchase_orders.count()}")

print(f"{'Total Audit Records':<30}: {duplicate_purchase_orders.count() + invalid_purchase_orders.count()}")

print(f"{'Bronze Records':<30}: {spark.table(BRONZE_PURCHASE_ORDERS).count()}")